# COROSred: Experiment 0 Evaluation

evaluates the Phase A Reliability Head to guarantee that routing tokens based on correctness $h_{i-1}$ successfully uncouples genuine topological error from simple next-token entropy.

**Gates for progressing to Phase B:**
1. High-Entropy Subset $\Delta$AUC (Head vs. Raw Entropy) $\ge$ 0.05
2. Exposure bias between teacher-forcing and deployment metrics remains stable.

In [1]:
import os
import sys
import yaml
import mlx.core as mx
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "COROSred").exists() and project_root.parent != project_root:
    project_root = project_root.parent
os.chdir(project_root)
sys.path.insert(0, str(project_root))

from COROSred.model import COROSredTransformer
from COROSred.eval.evaluator import COROSredExperiment0Evaluator

print("Loading Phase A artifacts...")
with open("configs/corosred/phase_a.yaml", "r") as f:
    cfg = yaml.safe_load(f)

model = COROSredTransformer(**cfg["model"])
model.load_weights("checkpoints/corosred/phase_a_head/model.safetensors", strict=False)
model.eval()

Loading Phase A artifacts...


RuntimeError: [load_safetensors] Failed to open file checkpoints/corosred/phase_a_head/model.safetensors

In [ ]:
# Load actual token sequences from eval set here. 
# (For this notebook template, we generate 16 random sequences of len 256 just to test execution)
eval_data = mx.random.randint(0, cfg["model"]["vocab_size"], (32, 256))

# 1. Evaluate Stratified AUC (High/Low Entropy)
evaluator = COROSredExperiment0Evaluator(model, entropy_threshold=1.5, k_amb=cfg["corosred"]["k_amb"])

print("\n--- Test A: Teacher-Forced Token Reliability (Stratified) ---")
results = evaluator.evaluate_teacher_forced(eval_data)

print(f"High-Entropy Head AUC:    {results['high_entropy_head_auc']:.4f}")
print(f"High-Entropy Entropy AUC: {results['high_entropy_entropy_auc']:.4f}")
print(f"Low-Entropy Error Recall: {results['low_entropy_error_recall']:.4f}")
print(f"\nDelta AUC (Gate > 0.05):  {results['delta_auc']:.4f}")
print(f"PHASE A GATE STATUS:      {'PASSED' if results['phase_a_gate_passed'] else 'FAILED'}")

In [ ]:
# 2. Evaluate Out-Of-Distribution (OOD) Prose Corruption Generalization
print("\n--- Test B: OOD Synthetic Prose Corruption ---")
prose_results = evaluator.evaluate_prose_corruption(eval_data, corruption_prob=0.05)
print(f"OOD Corruption Localization AUC: {prose_results['prose_localization_auc']:.4f}")
print(f"Total synthetic corruptions evaluated: {prose_results['total_corruptions_evaluated']}")